In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col,explode,sequence,to_date,lit,day,month,year,date_format,quarter,dayofweek,when, row_number
#Create DataFrame containing all dates from 2017 to 2021 for the dim_date dimension table
start_date = "2017-01-01"
end_date = "2021-12-31"

date_range = spark.range(1).select(
    explode(
        sequence(
            to_date(lit(start_date)),
            to_date(lit(end_date))
        )
    ).alias("Date")
)

#Create window to generate a surrogate key for each row in the date dimension table 
window = Window.orderBy("Date")

#Create dim_date dimension table with columns DateKey,DayName,Day,Month,Year,Quarter,IsWeekend,IsHoliday,Season
dim_date = date_range.withColumn("DateSK",row_number().over(window))\
    .withColumn("DayName",date_format("Date","EEEE"))\
    .withColumn("Day",day("Date"))\
    .withColumn("Month",month("Date"))\
    .withColumn("Year",year("Date"))\
    .withColumn("Quarter",quarter("Date"))\
    .withColumn("IsWeekend",when(dayofweek("Date").isin([1,7]),"True")
                .otherwise("False").cast("boolean"))\
    .withColumn("IsHoliday",when(((col("Month")==12) & (col("Day")==25)) 
                                | ((col("Month")==1) & (col("Day")==1)),"True")
                .otherwise("False").cast("boolean"))\
    .withColumn("Season",when(col("Month").isin([12,1,2]),"Winter")\
                .when(col("Month").isin([3,4,5]),"Spring")
                .when(col("Month").isin([6,7,8]),"Summer")
                .otherwise("Autumn"))
    

display(dim_date)


In [0]:
#Save as Delta Table in the gold layer
dim_date.write.format("delta").mode("overwrite").saveAsTable("accenture_final_project.gold_layer.dim_date")